In [3]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-3"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [1]:
## /public/trendytech/datasets/windowdata.csv

In [5]:
windowDF = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/public/trendytech/datasets/windowdata.csv")

## Windowing aggregations

In [16]:
windowDF.sort("country").show()

+---------------+-------+-----------+-------------+------------+
|        country|weeknum|numinvoices|totalquantity|invoicevalue|
+---------------+-------+-----------+-------------+------------+
|      Australia|     49|          1|          214|       258.9|
|      Australia|     48|          1|          107|      358.25|
|      Australia|     50|          2|          133|      387.95|
|        Austria|     50|          2|            3|      257.04|
|        Bahrain|     51|          1|           54|      205.74|
|        Belgium|     48|          1|          528|       346.1|
|        Belgium|     50|          2|          285|      625.16|
|        Belgium|     51|          2|          942|      838.65|
|Channel Islands|     49|          1|           80|      363.53|
|         Cyprus|     50|          1|          917|     1590.82|
|        Denmark|     49|          1|          454|      1281.5|
|        Finland|     50|          1|         1254|       892.8|
|         France|     49|

#### running total over a window

#### Window.partitionBy("").orderBy("").rowsBetween(,)

In [9]:
# 1. partition by country
# 2. sort by week num
# 3. define window size

In [33]:
# define a window - need 1.partition column, sort by column, and the rows to be selected

In [12]:
from pyspark.sql import *  ## needed for Window.

In [13]:
mywindow = Window.partitionBy("country").orderBy("weeknum").rowsBetween(Window.unboundedPreceding,Window.currentRow)

In [22]:
#### start from the very first row of a country's partition, and go up to (and including) the current row."

#### over(mywindow)

In [14]:
res_df = windowDF.withColumn("running_tot",sum("invoicevalue").over(mywindow))

In [15]:
res_df.show(20)

+-------+-------+-----------+-------------+------------+------------------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|       running_tot|
+-------+-------+-----------+-------------+------------+------------------+
| Sweden|     50|          3|         3714|      2646.3|            2646.3|
|Germany|     48|         11|         1795|     3309.75|           3309.75|
|Germany|     49|         12|         1852|     4521.39|           7831.14|
|Germany|     50|         15|         1973|     5065.79|          12896.93|
|Germany|     51|          5|         1103|     1665.91|          14562.84|
| France|     48|          4|         1299|     2808.16|           2808.16|
| France|     49|          9|         2303|     4527.01|           7335.17|
| France|     50|          6|          529|      537.32|           7872.49|
| France|     51|          5|          847|     1702.87|           9575.36|
|Belgium|     48|          1|          528|       346.1|             346.1|
|Belgium|   

In [17]:
mywindow1= Window.partitionBy("country").orderBy("weeknum").rowsBetween(-1,Window.currentRow)

In [21]:
## sum only the previous row and current row

In [18]:
res_df1 = windowDF.withColumn("running_tot",sum("invoicevalue").over(mywindow1))

In [19]:
res_df1.show()

+-------+-------+-----------+-------------+------------+------------------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|       running_tot|
+-------+-------+-----------+-------------+------------+------------------+
| Sweden|     50|          3|         3714|      2646.3|            2646.3|
|Germany|     48|         11|         1795|     3309.75|           3309.75|
|Germany|     49|         12|         1852|     4521.39|           7831.14|
|Germany|     50|         15|         1973|     5065.79|           9587.18|
|Germany|     51|          5|         1103|     1665.91|            6731.7|
| France|     48|          4|         1299|     2808.16|           2808.16|
| France|     49|          9|         2303|     4527.01|           7335.17|
| France|     50|          6|          529|      537.32|           5064.33|
| France|     51|          5|          847|     1702.87|           2240.19|
|Belgium|     48|          1|          528|       346.1|             346.1|
|Belgium|   

In [23]:
mywindow3 = Window.partitionBy("country").orderBy("weeknum").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)

In [24]:
# Grand total per country on every row (no "running", full sum repeated)

In [25]:
res_df2 = windowDF.withColumn("country_tot",sum("invoicevalue").over(mywindow3))

In [26]:
res_df2.show()

+-------+-------+-----------+-------------+------------+------------------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|       country_tot|
+-------+-------+-----------+-------------+------------+------------------+
| Sweden|     50|          3|         3714|      2646.3|            2646.3|
|Germany|     48|         11|         1795|     3309.75|          14562.84|
|Germany|     49|         12|         1852|     4521.39|          14562.84|
|Germany|     50|         15|         1973|     5065.79|          14562.84|
|Germany|     51|          5|         1103|     1665.91|          14562.84|
| France|     48|          4|         1299|     2808.16|           9575.36|
| France|     49|          9|         2303|     4527.01|           9575.36|
| France|     50|          6|          529|      537.32|           9575.36|
| France|     51|          5|          847|     1702.87|           9575.36|
|Belgium|     48|          1|          528|       346.1|1809.9099999999999|
|Belgium|   

In [27]:
mywindow4 = Window.partitionBy("country").orderBy("weeknum").rowsBetween(-1,1)

In [28]:
#Just a 3-row moving window: current row + 1 before + 1 after

In [31]:
res_df3 = windowDF.withColumn("tot",sum("invoicevalue").over(mywindow4))

In [32]:
res_df3.show()

+-------+-------+-----------+-------------+------------+------------------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|               tot|
+-------+-------+-----------+-------------+------------+------------------+
| Sweden|     50|          3|         3714|      2646.3|            2646.3|
|Germany|     48|         11|         1795|     3309.75|           7831.14|
|Germany|     49|         12|         1852|     4521.39|          12896.93|
|Germany|     50|         15|         1973|     5065.79|          11253.09|
|Germany|     51|          5|         1103|     1665.91|            6731.7|
| France|     48|          4|         1299|     2808.16|           7335.17|
| France|     49|          9|         2303|     4527.01|           7872.49|
| France|     50|          6|          529|      537.32|            6767.2|
| France|     51|          5|          847|     1702.87|           2240.19|
|Belgium|     48|          1|          528|       346.1|            971.26|
|Belgium|   

### rangeBetween()

In [34]:
# rangeBetween: all rows whose weeknum value falls within [current-1, current]
# if two rows share the same weeknum, BOTH get included, even though
# that's "more than 1 physical row"
# Window.orderBy("weeknum").rangeBetween(-1, 0)

In [35]:
# For your running-total case they'd behave identically only because weeknum has no duplicates within a country. 
# If two rows in Australia both had weeknum = 50,
# rangeBetween would include both of them at that point in the running sum, 
# while rowsBetween would treat them as two distinct sequential steps